In [1]:
!pwd

/ix/cs2770_2026s/abn80/cs2770_project/notebooks


In [2]:
"""
Explore LLaVA-1.5-7B Vision Encoder Architecture
==================================================
Run on HPC: sbatch or srun with GPU (any GPU fine, model loads in fp16 ~14GB)

Purpose: Before writing diffusion pretraining code, we need to understand:
  1. Exact ViT-L/14 layer structure (num layers, heads, hidden dim, MLP dim)
  2. Patch embedding details (patch size, image resolution, num patches, stride)
  3. Positional embedding shape and type (learned vs sinusoidal)
  4. CLS token presence and handling
  5. Projection MLP architecture (input dim → output dim, num layers)
  6. Which parameters are where (for freezing/unfreezing decisions)
  7. Feature shapes at each stage (for designing the denoising head)

Usage:
  conda activate medvlm
  python explore_vision_encoder.py
"""

import torch
from transformers import LlavaForConditionalGeneration, AutoProcessor
from collections import OrderedDict

MODEL_ID = "../models/llava-1.5-7b-hf"

def main():
    print("=" * 80)
    print("LOADING MODEL")
    print("=" * 80)

    model = LlavaForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
    # Don't move to GPU — we're just inspecting architecture, CPU is fine
    for name, child in model.model.named_children():
        print(name, type(child).__name__)
    processor = AutoProcessor.from_pretrained(MODEL_ID)

    # =========================================================================
    # 1. TOP-LEVEL MODEL STRUCTURE
    # =========================================================================
    print("\n" + "=" * 80)
    print("1. TOP-LEVEL MODEL STRUCTURE")
    print("=" * 80)
    for name, child in model.named_children():
        num_params = sum(p.numel() for p in child.parameters())
        print(f"  {name}: {child.__class__.__name__} | {num_params/1e6:.1f}M params")

    # =========================================================================
    # 2. VISION TOWER (CLIP ViT-L/14) DETAILED ARCHITECTURE
    # =========================================================================
    print("\n" + "=" * 80)
    print("2. VISION TOWER ARCHITECTURE")
    print("=" * 80)

    vision_tower = model.model.vision_tower
    vision_model = vision_tower.vision_model  # The actual CLIPVisionModel

    print(f"\n  Vision tower class: {vision_tower.__class__.__name__}")
    print(f"  Vision model class: {vision_model.__class__.__name__}")

    # Config
    config = vision_model.config
    print(f"\n  --- Config ---")
    print(f"  hidden_size:             {config.hidden_size}")
    print(f"  intermediate_size:       {config.intermediate_size}")
    print(f"  num_hidden_layers:       {config.num_hidden_layers}")
    print(f"  num_attention_heads:     {config.num_attention_heads}")
    print(f"  image_size:              {config.image_size}")
    print(f"  patch_size:              {config.patch_size}")
    print(f"  projection_dim:          {getattr(config, 'projection_dim', 'N/A')}")
    print(f"  hidden_act:              {config.hidden_act}")
    print(f"  layer_norm_eps:          {config.layer_norm_eps}")

    num_patches_per_side = config.image_size // config.patch_size
    num_patches = num_patches_per_side ** 2
    print(f"\n  --- Derived ---")
    print(f"  num_patches_per_side:    {num_patches_per_side}")
    print(f"  num_patches (no CLS):    {num_patches}")
    print(f"  total_seq_len (w/ CLS):  {num_patches + 1}")

    # =========================================================================
    # 3. PATCH EMBEDDING LAYER
    # =========================================================================
    print("\n" + "=" * 80)
    print("3. PATCH EMBEDDING LAYER")
    print("=" * 80)

    embeddings = vision_model.embeddings
    print(f"\n  Embeddings class: {embeddings.__class__.__name__}")

    # Patch embedding (Conv2d)
    patch_emb = embeddings.patch_embedding
    print(f"\n  patch_embedding: {patch_emb}")
    print(f"    in_channels:  {patch_emb.in_channels}")
    print(f"    out_channels: {patch_emb.out_channels}")
    print(f"    kernel_size:  {patch_emb.kernel_size}")
    print(f"    stride:       {patch_emb.stride}")
    print(f"    weight shape: {patch_emb.weight.shape}")

    # Class embedding (CLS token)
    cls_emb = embeddings.class_embedding
    print(f"\n  class_embedding (CLS token):")
    print(f"    shape: {cls_emb.shape}")
    print(f"    dtype: {cls_emb.dtype}")
    print(f"    requires_grad: {cls_emb.requires_grad}")

    # Position embedding
    pos_emb = embeddings.position_embedding
    print(f"\n  position_embedding:")
    print(f"    type:   {pos_emb.__class__.__name__}")
    print(f"    weight shape: {pos_emb.weight.shape}")
    print(f"    num_embeddings: {pos_emb.num_embeddings}")
    print(f"    embedding_dim:  {pos_emb.embedding_dim}")

    # Position IDs
    if hasattr(embeddings, 'position_ids'):
        print(f"    position_ids shape: {embeddings.position_ids.shape}")
        print(f"    position_ids range: [{embeddings.position_ids.min()}, {embeddings.position_ids.max()}]")

    # =========================================================================
    # 4. TRANSFORMER ENCODER LAYERS
    # =========================================================================
    print("\n" + "=" * 80)
    print("4. TRANSFORMER ENCODER LAYERS")
    print("=" * 80)

    encoder = vision_model.encoder
    layers = encoder.layers
    print(f"\n  Number of encoder layers: {len(layers)}")

    # Inspect first layer in detail
    layer0 = layers[0]
    print(f"\n  --- Layer 0 detailed structure ---")
    for name, module in layer0.named_modules():
        if name == '':
            continue
        param_count = sum(p.numel() for p in module.parameters(recurse=False))
        if param_count > 0:
            print(f"    {name}: {module.__class__.__name__} | {param_count/1e3:.1f}K params")

    # Self-attention details
    attn = layer0.self_attn
    print(f"\n  --- Self-Attention (layer 0) ---")
    print(f"    num_heads:    {attn.num_heads}")
    print(f"    head_dim:     {attn.head_dim}")
    print(f"    embed_dim:    {attn.embed_dim}")
    print(f"    q_proj:       {attn.q_proj.weight.shape}")
    print(f"    k_proj:       {attn.k_proj.weight.shape}")
    print(f"    v_proj:       {attn.v_proj.weight.shape}")
    print(f"    out_proj:     {attn.out_proj.weight.shape}")

    # MLP details
    mlp = layer0.mlp
    print(f"\n  --- MLP (layer 0) ---")
    print(f"    fc1:          {mlp.fc1.weight.shape}")
    print(f"    fc2:          {mlp.fc2.weight.shape}")
    print(f"    activation:   {mlp.activation_fn}")

    # Layer norms
    print(f"\n  --- Layer Norms (layer 0) ---")
    print(f"    layer_norm1:  weight shape = {layer0.layer_norm1.weight.shape}")
    print(f"    layer_norm2:  weight shape = {layer0.layer_norm2.weight.shape}")

    # =========================================================================
    # 5. POST-ENCODER LAYER NORM & FEATURE SELECTION
    # =========================================================================
    print("\n" + "=" * 80)
    print("5. POST-ENCODER PROCESSING")
    print("=" * 80)

    # Check for pre/post layernorm
    if hasattr(vision_model, 'pre_layrnorm'):
        print(f"  pre_layrnorm: {vision_model.pre_layrnorm}")
    if hasattr(vision_model, 'post_layernorm'):
        print(f"  post_layernorm: {vision_model.post_layernorm}")
        print(f"    weight shape: {vision_model.post_layernorm.weight.shape}")

    # Check LLaVA's vision_feature_select_strategy
    llava_config = model.config
    print(f"\n  LLaVA vision_feature_layer:          {getattr(llava_config, 'vision_feature_layer', 'N/A')}")
    print(f"  LLaVA vision_feature_select_strategy: {getattr(llava_config, 'vision_feature_select_strategy', 'N/A')}")

    # =========================================================================
    # 6. MULTI-MODAL PROJECTOR (PROJECTION MLP)
    # =========================================================================
    print("\n" + "=" * 80)
    print("6. MULTI-MODAL PROJECTOR (Projection MLP)")
    print("=" * 80)

    projector = model.model.multi_modal_projector
    print(f"\n  Projector class: {projector.__class__.__name__}")
    print(f"\n  Full structure:")
    for name, module in projector.named_modules():
        if name == '':
            continue
        if hasattr(module, 'weight'):
            print(f"    {name}: {module.__class__.__name__}")
            print(f"      weight shape: {module.weight.shape}")
            if hasattr(module, 'bias') and module.bias is not None:
                print(f"      bias shape:   {module.bias.shape}")

    proj_params = sum(p.numel() for p in projector.parameters())
    print(f"\n  Total projector params: {proj_params/1e6:.2f}M")

    # =========================================================================
    # 7. FORWARD PASS SHAPE TRACE (dummy input)
    # =========================================================================
    print("\n" + "=" * 80)
    print("7. FORWARD PASS SHAPE TRACE")
    print("=" * 80)

    # Create dummy image
    from PIL import Image
    import numpy as np
    dummy_img = Image.fromarray(np.random.randint(0, 255, (336, 336, 3), dtype=np.uint8))

    # Process through the image processor
    inputs = processor(images=dummy_img,text='dummy',return_tensors="pt")
    pixel_values = inputs['pixel_values']  # already preprocessed
    print(f"\n  Input pixel_values shape:  {pixel_values.shape}")
    print(f"  Input pixel_values dtype:  {pixel_values.dtype}")
    print(f"  Input pixel_values range:  [{pixel_values.min():.3f}, {pixel_values.max():.3f}]")

    # Manual forward through vision encoder
    pixel_values = pixel_values.to(torch.float16)

    with torch.no_grad():
        # Step 1: Patch embedding
        patch_embeds = embeddings.patch_embedding(pixel_values)
        print(f"\n  After patch_embedding (Conv2d): {patch_embeds.shape}")

        patch_embeds = patch_embeds.flatten(2).transpose(1, 2)
        print(f"  After flatten+transpose:        {patch_embeds.shape}")

        # Step 2: Add CLS + position embeddings
        batch_size = patch_embeds.shape[0]
        cls_tokens = cls_emb.expand(batch_size, 1, -1).to(patch_embeds.dtype)
        embeddings_out = torch.cat([cls_tokens, patch_embeds], dim=1)
        print(f"  After prepending CLS token:     {embeddings_out.shape}")

        position_ids = embeddings.position_ids[:, :embeddings_out.shape[1]]
        embeddings_out = embeddings_out + pos_emb(position_ids)
        print(f"  After adding position emb:      {embeddings_out.shape}")

        # Step 3: Full vision model forward
        vision_outputs = vision_model(pixel_values, output_hidden_states=True)

        print(f"\n  --- Vision model outputs ---")
        print(f"  last_hidden_state shape: {vision_outputs.last_hidden_state.shape}")
        print(f"  pooler_output shape:     {vision_outputs.pooler_output.shape if vision_outputs.pooler_output is not None else 'None'}")
        print(f"  num hidden_states:       {len(vision_outputs.hidden_states)}")

        # Show shape of each hidden state (first few + last few)
        print(f"\n  Hidden state shapes (layer → shape):")
        for i, hs in enumerate(vision_outputs.hidden_states):
            if i < 3 or i >= len(vision_outputs.hidden_states) - 3:
                print(f"    Layer {i:2d}: {hs.shape}")
            elif i == 3:
                print(f"    ...")

        # Step 4: Feature selection (what LLaVA actually uses)
        # LLaVA-1.5 uses features from layer -2 (second-to-last), excluding CLS
        feature_layer = getattr(llava_config, 'vision_feature_layer', -2)
        select_strategy = getattr(llava_config, 'vision_feature_select_strategy', 'default')

        selected_features = vision_outputs.hidden_states[feature_layer]
        print(f"\n  Selected feature layer ({feature_layer}): {selected_features.shape}")

        if select_strategy == 'default':
            # Remove CLS token
            selected_features = selected_features[:, 1:]
            print(f"  After removing CLS (strategy='{select_strategy}'): {selected_features.shape}")

        # Step 5: Through projection MLP
        projected = projector(selected_features.to(torch.float16))
        print(f"  After projection MLP:           {projected.shape}")

    # =========================================================================
    # 8. PARAMETER GROUPS SUMMARY (for pretraining decisions)
    # =========================================================================
    print("\n" + "=" * 80)
    print("8. PARAMETER GROUPS SUMMARY")
    print("=" * 80)

    groups = OrderedDict()
    for name, param in model.named_parameters():
        # Group by top-level component
        parts = name.split('.')
        if 'vision_tower' in name:
            if 'embeddings' in name:
                key = 'vision_tower.embeddings'
            elif 'encoder.layers' in name:
                layer_idx = name.split('encoder.layers.')[1].split('.')[0]
                key = f'vision_tower.encoder.layer_{layer_idx}'
            elif 'post_layernorm' in name or 'pre_layrnorm' in name:
                key = 'vision_tower.layernorm'
            else:
                key = 'vision_tower.other'
        elif 'multi_modal_projector' in name:
            key = 'multi_modal_projector'
        elif 'language_model' in name:
            key = 'language_model'
        else:
            key = 'other'

        if key not in groups:
            groups[key] = {'count': 0, 'numel': 0}
        groups[key]['count'] += 1
        groups[key]['numel'] += param.numel()

    print(f"\n  {'Component':<35} {'#Params':>12} {'#Tensors':>10}")
    print(f"  {'-'*35} {'-'*12} {'-'*10}")
    total = 0
    for key, info in groups.items():
        if key == 'language_model':
            print(f"  {key:<35} {info['numel']/1e6:>10.2f}M {info['count']:>10}")
        else:
            print(f"  {key:<35} {info['numel']/1e6:>10.2f}M {info['count']:>10}")
        total += info['numel']
    print(f"  {'TOTAL':<35} {total/1e6:>10.2f}M")

    # =========================================================================
    # 9. KEY TAKEAWAYS FOR DIFFUSION PRETRAINING
    # =========================================================================
    print("\n" + "=" * 80)
    print("9. KEY TAKEAWAYS FOR DIFFUSION PRETRAINING")
    print("=" * 80)
    print(f"""
  ARCHITECTURE SUMMARY:
    - ViT-L/14: {config.num_hidden_layers} layers, {config.num_attention_heads} heads, hidden_dim={config.hidden_size}
    - Image: {config.image_size}x{config.image_size} → {num_patches_per_side}x{num_patches_per_side} = {num_patches} patches
    - Patch size: {config.patch_size}x{config.patch_size}
    - Sequence: [CLS] + {num_patches} patch tokens = {num_patches+1} total tokens
    - Each token: {config.hidden_size}-dimensional

  FOR DIFFUSION DENOISING HEAD:
    - Input dim:  {config.hidden_size} (from ViT hidden states)
    - Spatial:    {num_patches_per_side}x{num_patches_per_side} grid of patches
    - You can denoise in patch-embedding space ({config.hidden_size}-dim per patch)
    - Or reshape to spatial grid and denoise in pixel space

  FOR TIMESTEP CONDITIONING:
    - Add sinusoidal timestep embedding to patch tokens
    - Dimension should match hidden_size = {config.hidden_size}
    - Add BEFORE the transformer layers (alongside position embeddings)
    - Or add at each layer (like adaptive layer norm in DiT)

  FOR REINTEGRATION INTO LLAVA:
    - LLaVA uses hidden states from layer {feature_layer} (second-to-last)
    - CLS token is REMOVED before projection
    - Projection input:  {num_patches} tokens × {config.hidden_size} dim
    - Projection output: {num_patches} tokens × {projected.shape[-1]} dim
    - The projection MLP MUST be realigned after pretraining
""")

    print("=" * 80)
    print("EXPLORATION COMPLETE")
    print("=" * 80)


if __name__ == "__main__":
    main()

LOADING MODEL


Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

vision_tower CLIPVisionModel
multi_modal_projector LlavaMultiModalProjector
language_model LlamaModel

1. TOP-LEVEL MODEL STRUCTURE
  model: LlavaModel | 6932.1M params
  lm_head: Linear | 131.3M params

2. VISION TOWER ARCHITECTURE

  Vision tower class: CLIPVisionModel
  Vision model class: CLIPVisionTransformer

  --- Config ---
  hidden_size:             1024
  intermediate_size:       4096
  num_hidden_layers:       24
  num_attention_heads:     16
  image_size:              336
  patch_size:              14
  projection_dim:          768
  hidden_act:              quick_gelu
  layer_norm_eps:          1e-05

  --- Derived ---
  num_patches_per_side:    24
  num_patches (no CLS):    576
  total_seq_len (w/ CLS):  577

3. PATCH EMBEDDING LAYER

  Embeddings class: CLIPVisionEmbeddings

  patch_embedding: Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
    in_channels:  3
    out_channels: 1024
    kernel_size:  (14, 14)
    stride:       (14, 14)
    weight shape

## Explore Medtrinity Dataset


In [6]:
"""
Explore MedTrinity-25M Dataset Image Properties
================================================
Run on HPC with internet access, or after downloading a subset.

This script samples images from MedTrinity and reports:
  1. Image size distribution (width, height)
  2. Aspect ratios
  3. Color channels (RGB vs grayscale)
  4. Modality distribution (from metadata)
  5. File format stats
  6. What resizing to 336x336 would look like

Usage:
  conda activate medvlm
  python explore_medtrinity.py
"""

import os
import json
import numpy as np
from collections import Counter, defaultdict
from PIL import Image
from datasets import load_dataset
import time

# =========================================================================
# CONFIG
# =========================================================================
# MedTrinity is huge (25M), so we stream and sample
NUM_SAMPLES = 5000  # Sample this many to get stats
SEED = 42

def main():
    print("=" * 80)
    print("EXPLORING MEDTRINITY-25M DATASET")
    print("=" * 80)

    # =====================================================================
    # 1. Load dataset in streaming mode (no full download needed)
    # =====================================================================
    print("\n[1/6] Loading dataset in streaming mode...")
    print("  This streams from HuggingFace without downloading everything.")

    ds = load_dataset(
        "UCSC-VLAA/MedTrinity-25M", '25M_demo',
        split="train",
        streaming=True,
    )

    # =====================================================================
    # 2. Sample and collect stats
    # =====================================================================
    print(f"\n[2/6] Sampling {NUM_SAMPLES} images...")

    widths = []
    heights = []
    aspect_ratios = []
    channels = []
    modes = []
    modalities = []
    caption_lengths = []
    pixel_areas = []

    np.random.seed(SEED)
    count = 0
    start = time.time()

    for i, sample in enumerate(ds):
        if count >= NUM_SAMPLES:
            break

        try:
            img = sample.get("image", None)
            if img is None:
                continue

            # If image is already a PIL Image (HF datasets handles this)
            if not isinstance(img, Image.Image):
                continue

            w, h = img.size
            widths.append(w)
            heights.append(h)
            aspect_ratios.append(w / h)
            modes.append(img.mode)
            channels.append(len(img.getbands()))
            pixel_areas.append(w * h)

            # Check for metadata
            if "modality" in sample:
                modalities.append(sample["modality"])
            elif "metadata" in sample and isinstance(sample["metadata"], dict):
                modalities.append(sample["metadata"].get("modality", "unknown"))
            elif "caption" in sample:
                cap = sample["caption"]
                if isinstance(cap, str):
                    caption_lengths.append(len(cap.split()))

            count += 1
            if count % 500 == 0:
                elapsed = time.time() - start
                print(f"    Processed {count}/{NUM_SAMPLES} ({elapsed:.1f}s)")

        except Exception as e:
            print(f"    Error at sample {i}: {e}")
            continue

    elapsed = time.time() - start
    print(f"  Done! Processed {count} samples in {elapsed:.1f}s")

    # =====================================================================
    # 3. Image size statistics
    # =====================================================================
    print("\n" + "=" * 80)
    print("[3/6] IMAGE SIZE STATISTICS")
    print("=" * 80)

    widths = np.array(widths)
    heights = np.array(heights)
    areas = np.array(pixel_areas)
    ratios = np.array(aspect_ratios)

    print(f"\n  Width:")
    print(f"    min:    {widths.min()}")
    print(f"    max:    {widths.max()}")
    print(f"    mean:   {widths.mean():.1f}")
    print(f"    median: {np.median(widths):.1f}")
    print(f"    std:    {widths.std():.1f}")

    print(f"\n  Height:")
    print(f"    min:    {heights.min()}")
    print(f"    max:    {heights.max()}")
    print(f"    mean:   {heights.mean():.1f}")
    print(f"    median: {np.median(heights):.1f}")
    print(f"    std:    {heights.std():.1f}")

    print(f"\n  Aspect ratio (w/h):")
    print(f"    min:    {ratios.min():.3f}")
    print(f"    max:    {ratios.max():.3f}")
    print(f"    mean:   {ratios.mean():.3f}")
    print(f"    median: {np.median(ratios):.3f}")

    print(f"\n  Pixel area:")
    print(f"    min:    {areas.min():,}")
    print(f"    max:    {areas.max():,}")
    print(f"    mean:   {areas.mean():,.0f}")

    # Size buckets
    print(f"\n  Size distribution:")
    size_buckets = Counter()
    for w, h in zip(widths, heights):
        s = max(w, h)
        if s < 128:
            size_buckets["< 128px"] += 1
        elif s < 256:
            size_buckets["128-255px"] += 1
        elif s < 336:
            size_buckets["256-335px"] += 1
        elif s < 512:
            size_buckets["336-511px"] += 1
        elif s < 1024:
            size_buckets["512-1023px"] += 1
        else:
            size_buckets[">= 1024px"] += 1

    for bucket in ["< 128px", "128-255px", "256-335px", "336-511px", "512-1023px", ">= 1024px"]:
        n = size_buckets.get(bucket, 0)
        pct = 100 * n / count
        bar = "#" * int(pct / 2)
        print(f"    {bucket:>14s}: {n:5d} ({pct:5.1f}%) {bar}")

    # =====================================================================
    # 4. Color channel statistics
    # =====================================================================
    print("\n" + "=" * 80)
    print("[4/6] COLOR CHANNEL STATISTICS")
    print("=" * 80)

    mode_counts = Counter(modes)
    for mode, n in mode_counts.most_common():
        pct = 100 * n / count
        print(f"    {mode:>6s}: {n:5d} ({pct:5.1f}%)")

    channel_counts = Counter(channels)
    print(f"\n  Channel distribution:")
    for ch, n in sorted(channel_counts.items()):
        pct = 100 * n / count
        print(f"    {ch} channels: {n:5d} ({pct:5.1f}%)")

    # =====================================================================
    # 5. Modality distribution (if available)
    # =====================================================================
    print("\n" + "=" * 80)
    print("[5/6] MODALITY DISTRIBUTION")
    print("=" * 80)

    if modalities:
        mod_counts = Counter(modalities)
        for mod, n in mod_counts.most_common(20):
            pct = 100 * n / count
            print(f"    {mod:>30s}: {n:5d} ({pct:5.1f}%)")
    else:
        print("  No modality metadata found in sampled records.")

    if caption_lengths:
        cap_lens = np.array(caption_lengths)
        print(f"\n  Caption length (words):")
        print(f"    min:    {cap_lens.min()}")
        print(f"    max:    {cap_lens.max()}")
        print(f"    mean:   {cap_lens.mean():.1f}")
        print(f"    median: {np.median(cap_lens):.1f}")

    # =====================================================================
    # 6. Implications for our preprocessing
    # =====================================================================
    print("\n" + "=" * 80)
    print("[6/6] IMPLICATIONS FOR DIFFUSION PRETRAINING")
    print("=" * 80)

    too_small = np.sum((widths < 100) | (heights < 100))
    needs_upscale = np.sum((widths < 336) & (heights < 336))
    already_ok = np.sum((widths >= 336) & (heights >= 336))

    print(f"\n  Target resolution: 336 x 336 (LLaVA's CLIP ViT-L/14)")
    print(f"  Images < 100px on either side:     {too_small:5d} ({100*too_small/count:.1f}%) — consider filtering")
    print(f"  Images smaller than 336x336:        {needs_upscale:5d} ({100*needs_upscale/count:.1f}%) — need upscaling")
    print(f"  Images already >= 336x336:          {already_ok:5d} ({100*already_ok/count:.1f}%) — just resize/crop")
    print(f"\n  Recommended preprocessing:")
    print(f"    1. Filter out images < 100px on any side")
    print(f"    2. Use CLIP's standard transform:")
    print(f"       - Resize shortest side to 336")
    print(f"       - Center crop to 336x336")
    print(f"       - Normalize with CLIP mean/std")
    print(f"    3. Convert grayscale to RGB (repeat channels)")

    # Show what CLIP's processor expects
    print(f"\n  For reference, CLIP's image processor does:")
    print(f"    - Resize to (336, 336)")
    print(f"    - Normalize: mean=[0.48145466, 0.4578275, 0.40821073]")
    print(f"    -            std=[0.26862954, 0.26130258, 0.27577711]")

    # =====================================================================
    # 7. Sample data structure
    # =====================================================================
    print("\n" + "=" * 80)
    print("[BONUS] SAMPLE DATA STRUCTURE")
    print("=" * 80)

    ds2 = load_dataset("UCSC-VLAA/MedTrinity-25M", "25M_demo",split="train", streaming=True)
    sample = next(iter(ds2))
    print(f"\n  Available keys: {list(sample.keys())}")
    for k, v in sample.items():
        if k == "image":
            print(f"    {k}: PIL Image, size={v.size}, mode={v.mode}")
        elif isinstance(v, str):
            print(f"    {k}: str, length={len(v)}, preview='{v[:100]}...'")
        elif isinstance(v, dict):
            print(f"    {k}: dict, keys={list(v.keys())}")
        else:
            print(f"    {k}: {type(v).__name__}, value={v}")

    print("\n" + "=" * 80)
    print("EXPLORATION COMPLETE")
    print("=" * 80)


if __name__ == "__main__":
    main()

EXPLORING MEDTRINITY-25M DATASET

[1/6] Loading dataset in streaming mode...
  This streams from HuggingFace without downloading everything.


Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]


[2/6] Sampling 5000 images...
    Processed 500/5000 (0.8s)
    Processed 1000/5000 (1.0s)
    Processed 1500/5000 (1.3s)
    Processed 2000/5000 (1.6s)
    Processed 2500/5000 (1.9s)
    Processed 3000/5000 (2.3s)
    Processed 3500/5000 (2.6s)
    Processed 4000/5000 (2.9s)
    Processed 4500/5000 (3.1s)
    Processed 5000/5000 (3.4s)
  Done! Processed 5000 samples in 3.4s

[3/6] IMAGE SIZE STATISTICS

  Width:
    min:    512
    max:    512
    mean:   512.0
    median: 512.0
    std:    0.0

  Height:
    min:    512
    max:    512
    mean:   512.0
    median: 512.0
    std:    0.0

  Aspect ratio (w/h):
    min:    1.000
    max:    1.000
    mean:   1.000
    median: 1.000

  Pixel area:
    min:    262,144
    max:    262,144
    mean:   262,144

  Size distribution:
           < 128px:     0 (  0.0%) 
         128-255px:     0 (  0.0%) 
         256-335px:     0 (  0.0%) 
         336-511px:     0 (  0.0%) 
        512-1023px:  5000 (100.0%) ################################

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]


  Available keys: ['image', 'id', 'caption']
    image: PIL Image, size=(512, 512), mode=RGB
    id: str, length=36, preview='8031efe0-1b5c-11ef-8929-000066532cad...'
    caption: str, length=626, preview='The image is a non-contrasted computed tomography (CT) scan of the brain, showing the cerebral struc...'

EXPLORATION COMPLETE


### Download medtrinity locally

In [7]:
"""
Download & Save MedTrinity-25M Demo Subset for Training
========================================================
Downloads the demo split, preprocesses images to 336x336 using
CLIP's processor, and saves in a format ready for DataLoader.

Two output formats:
  1. HuggingFace dataset (arrow format) — fast random access for training
  2. Metadata CSV with image paths — for inspection/debugging

Usage:
  conda activate medvlm
  python save_medtrinity_demo.py
"""

import os
import time
from datasets import load_dataset
from transformers import AutoProcessor

# =========================================================================
# CONFIG — adjust paths to match your project layout
# =========================================================================
MODEL_PATH = "../models/llava-1.5-7b-hf"  # for CLIP processor
SAVE_DIR = "../data/medtrinity-demo"
HF_SAVE_PATH = os.path.join(SAVE_DIR, "hf_dataset")
IMAGES_DIR = os.path.join(SAVE_DIR, "images")  # raw images for debugging
DATASET_NAME = "UCSC-VLAA/MedTrinity-25M"
CONFIG_NAME = "25M_demo"

def main():
    print("=" * 80)
    print("DOWNLOADING MEDTRINITY-25M DEMO SUBSET")
    print("=" * 80)

    os.makedirs(SAVE_DIR, exist_ok=True)
    os.makedirs(IMAGES_DIR, exist_ok=True)

    # =====================================================================
    # 1. Download full demo split (not streaming — we want it all locally)
    # =====================================================================
    print("\n[1/4] Downloading demo split...")
    start = time.time()

    ds = load_dataset(
        DATASET_NAME,
        CONFIG_NAME,
        split="train",
    )

    elapsed = time.time() - start
    print(f"  Downloaded {len(ds)} samples in {elapsed:.1f}s")
    print(f"  Features: {ds.features}")
    print(f"  Sample keys: {ds.column_names}")

    # =====================================================================
    # 2. Verify data quality
    # =====================================================================
    print("\n[2/4] Verifying data quality...")

    bad_images = 0
    bad_captions = 0
    for i in range(min(1000, len(ds))):
        sample = ds[i]
        if sample["image"] is None:
            bad_images += 1
        if not sample["caption"] or len(sample["caption"].strip()) == 0:
            bad_captions += 1

    print(f"  Checked first {min(1000, len(ds))} samples:")
    print(f"    Bad images:   {bad_images}")
    print(f"    Bad captions: {bad_captions}")

    # =====================================================================
    # 3. Save as HuggingFace dataset (arrow format)
    #    This is the primary training format — fast random access
    # =====================================================================
    print(f"\n[3/4] Saving HuggingFace dataset to {HF_SAVE_PATH}...")
    start = time.time()

    ds.save_to_disk(HF_SAVE_PATH)

    elapsed = time.time() - start
    print(f"  Saved in {elapsed:.1f}s")

    # Verify we can reload it
    from datasets import load_from_disk
    ds_reloaded = load_from_disk(HF_SAVE_PATH)
    print(f"  Verified reload: {len(ds_reloaded)} samples")
    print(f"  Sample image size: {ds_reloaded[0]['image'].size}")
    print(f"  Sample caption preview: '{ds_reloaded[0]['caption'][:100]}...'")

    # =====================================================================
    # 4. Save a few sample images for visual inspection
    # =====================================================================
    print(f"\n[4/4] Saving 10 sample images to {IMAGES_DIR} for inspection...")

    for i in range(min(10, len(ds))):
        sample = ds[i]
        img = sample["image"]
        img_id = sample["id"][:8]  # first 8 chars of UUID
        img.save(os.path.join(IMAGES_DIR, f"sample_{i:04d}_{img_id}.png"))

        # Also save caption alongside
        with open(os.path.join(IMAGES_DIR, f"sample_{i:04d}_{img_id}.txt"), "w") as f:
            f.write(sample["caption"])

    print(f"  Saved 10 sample images + captions")

    # =====================================================================
    # Summary
    # =====================================================================
    print("\n" + "=" * 80)
    print("DONE!")
    print("=" * 80)
    print(f"\n  Dataset location:  {os.path.abspath(HF_SAVE_PATH)}")
    print(f"  Total samples:     {len(ds)}")
    print(f"  Image format:      512x512 RGB (resize to 336x336 during training)")
    print(f"  Columns:           image, id, caption")
    print(f"\n  To load for training:")
    print(f"    from datasets import load_from_disk")
    print(f"    ds = load_from_disk('{HF_SAVE_PATH}')")
    print(f"    img = ds[0]['image']   # PIL Image 512x512")
    print(f"    cap = ds[0]['caption'] # str")
    print(f"\n  Preprocessing during training:")
    print(f"    processor = AutoProcessor.from_pretrained('{MODEL_PATH}')")
    print(f"    inputs = processor(images=img, text='dummy', return_tensors='pt')")
    print(f"    pixel_values = inputs['pixel_values']  # [1, 3, 336, 336]")


if __name__ == "__main__":
    main()

DOWNLOADING MEDTRINITY-25M DEMO SUBSET

[1/4] Downloading demo split...


Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

data/train-00000-of-00010.parquet:   0%|          | 0.00/829M [00:00<?, ?B/s]

data/train-00001-of-00010.parquet:   0%|          | 0.00/764M [00:00<?, ?B/s]

data/train-00002-of-00010.parquet:   0%|          | 0.00/721M [00:00<?, ?B/s]

...

  Downloaded 161630 samples in 109.3s
  Features: {'image': Image(mode=None, decode=True), 'id': Value('string'), 'caption': Value('string')}
  Sample keys: ['image', 'id', 'caption']

[2/4] Verifying data quality...
  Checked first 1000 samples:
    Bad images:   0
    Bad captions: 0

[3/4] Saving HuggingFace dataset to ../data/medtrinity-demo/hf_dataset...


Saving the dataset (0/18 shards):   0%|          | 0/161630 [00:00<?, ? examples/s]

  Saved in 34.3s


Loading dataset from disk:   0%|          | 0/18 [00:00<?, ?it/s]

  Verified reload: 161630 samples
  Sample image size: (512, 512)
  Sample caption preview: 'The image is a non-contrasted computed tomography (CT) scan of the brain, showing the cerebral struc...'

[4/4] Saving 10 sample images to ../data/medtrinity-demo/images for inspection...
  Saved 10 sample images + captions

DONE!

  Dataset location:  /ix/cs2770_2026s/abn80/cs2770_project/data/medtrinity-demo/hf_dataset
  Total samples:     161630
  Image format:      512x512 RGB (resize to 336x336 during training)
  Columns:           image, id, caption

  To load for training:
    from datasets import load_from_disk
    ds = load_from_disk('../data/medtrinity-demo/hf_dataset')
    img = ds[0]['image']   # PIL Image 512x512
    cap = ds[0]['caption'] # str

  Preprocessing during training:
    processor = AutoProcessor.from_pretrained('../models/llava-1.5-7b-hf')
    inputs = processor(images=img, text='dummy', return_tensors='pt')
    pixel_values = inputs['pixel_values']  # [1, 3, 336, 336]